In [ ]:
import geopandas as gpd
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.patheffects as pe
import matplotlib.colors as mcolors

from skimage.measure import label
from tqdm.auto import tqdm

# Amazon Dataset Size

In [ ]:
with rasterio.open("/home/luizluz/Documentos/multi-task-fcn/amazon_input_data/segmentation/train_set.tif") as src:
    img_label = src.read()

unique_labels = np.unique(img_label)


In [ ]:
print("Quantidade de espécies:", unique_labels.shape)

In [ ]:
width = img_label.shape[-1]
height = img_label.shape[-2]
num_pixels = width * height
print(f"Width: {width}, Height: {height}, Number of pixels: {num_pixels:,}")


In [ ]:
comp = label(img_label)

In [ ]:
# For each unique label in img_label, count the number of connected components
from collections import defaultdict

component_counts = defaultdict(int)

for label_id in tqdm(unique_labels, ncols=100):
    # Skip background if needed (often 0)
    if label_id == 0:
        continue
    mask = (img_label == label_id)
    labeled = label(mask)
    # Exclude background (label 0)
    num_components = labeled.max()
    component_counts[label_id] = num_components

print("Components per id (label):")
for k, v in component_counts.items():
    print(f"Label {k}: {v} components")


In [ ]:
import pandas as pd
component_counts_df = pd.DataFrame(list(component_counts.items()), columns=['label', 'num_components'])
component_counts_df.sort_values(by='num_components', ascending=False)

In [ ]:
np.unique(comp).shape

# Bioflore Dataset Size

In [ ]:
gdf_path = "/home/luizluz/Documentos/multi-task-fcn/matematica_industria_data/raw/labels/labels.shp"
gdf = gpd.read_file(gdf_path)

gdf.plot()

In [ ]:
from glob import glob
import rasterio
tiff_paths = glob("../../matematica_industria_data/raw/geotiffs/*.tif")

datasets = [rasterio.open(tiff_path) for tiff_path in tiff_paths]

In [ ]:
from os.path import abspath
print(
    '"'+'" "'.join([abspath(tiff_path) for tiff_path in tiff_paths])+'"'
)


In [ ]:
# Compute the pixel count for each dataset (tiff)
pixel_counts = {}
for d in datasets:
    width = d.width
    height = d.height
    count = width * height
    pixel_counts[d.name] = count

# Print the pixel count per dataset
for tiff, count in pixel_counts.items():
    print(f"{tiff}: {count:,} pixels")

# Compute the total pixels across all tiffs
total_pixels = sum(pixel_counts.values())
print(f"Total pixels across all tiffs: {total_pixels:,}")


In [ ]:
import numpy as np
from shapely.geometry import box

# Get the CRS from the first raster (assuming all rasters share the same CRS)
raster_crs = datasets[0].crs

# Reproject gdf to match the raster CRS if they differ
if gdf.crs != raster_crs:
    gdf = gdf.to_crs(raster_crs)
    print(f"Reprojected GeoDataFrame from {gdf.crs} to {raster_crs}")

# Create a list of raster bounding boxes and their corresponding filenames
raster_infos = []
for d in datasets:
    bounds = d.bounds
    raster_bbox = box(bounds.left, bounds.bottom, bounds.right, bounds.top)
    raster_infos.append({'name': d.name, 'bbox': raster_bbox})

def find_raster_for_shape(shape):
    geom = shape.geometry
    for raster_info in raster_infos:
        if raster_info['bbox'].intersects(geom):
            return raster_info['name']
    return None

gdf['tiff_file'] = gdf.apply(find_raster_for_shape, axis=1)

In [ ]:
gdf['tiff_file'].value_counts()

In [ ]:
gdf.groupby('tiff_file')['species'].nunique()

In [ ]:
gdf["species"].value_counts()

In [ ]:
print("Quantidade de espécies:", gdf["species"].nunique())

In [ ]:
gdf.groupby("geotiff")["species"].apply(lambda x: np.sort(np.unique(x)))

# É possível trabalhar com apenas uma subamostra de todas as imagens que temos?

## Temos geotiffs que possuem exatamente as mesmas espécies?

In [ ]:
from itertools import combinations

# Cria um dicionário: geotiff -> set de espécies
geotiff_species = gdf.groupby("geotiff")["species"].apply(set).to_dict()

# Lista para registrar pares que possuem exatamente as mesmas espécies
geotiff_pairs_with_exact_species = []

# Testa todos os pares de geotiffs
for g1, g2 in combinations(geotiff_species.keys(), 2):
    if geotiff_species[g1] == geotiff_species[g2]:
        geotiff_pairs_with_exact_species.append((g1, g2, geotiff_species[g1]))

if geotiff_pairs_with_exact_species:
    print("Existem pares de geotiffs com exatamente as mesmas espécies:")
    for g1, g2, species_set in geotiff_pairs_with_exact_species:
        print(f"{g1} e {g2} possuem exatamente estas espécies: {sorted(list(species_set))}")
else:
    print("Não existem geotiffs diferentes contendo exatamente as mesmas espécies.")


## Temos um conjunto de geotiffs que possuem a mesma subamostra?


In [ ]:
gdf.groupby("geotiff")["species"].apply(lambda x: x.nunique()).sort_values(ascending=False)

In [ ]:
gdf.groupby("species")["geotiff"].apply(lambda x: x.nunique()).sort_values(ascending=False)

In [ ]:
gdf.groupby("species")["geotiff"].apply(lambda x: x.nunique()).sort_values(ascending=False).index[:5]

In [ ]:
# Definindo o conjunto das espécies de interesse
target_species = {
    'Tachigali aurea', 
    'Pterodon emarginatus', 
    'Qualea parviflora', 
    # 'Caryocar coriaceum', 
    'Salvertia convallariodora'
}

# Cria um dicionário: geotiff -> set de espécies (caso não tenha sido criado acima)
geotiff_species = gdf.groupby("geotiff")["species"].apply(set)

# Seleciona os geotiffs que possuem todas as espécies desejadas
geotiffs_with_all_target_species = geotiff_species[geotiff_species.apply(lambda s: target_species.issubset(s))]

print("Geotiffs que possuem TODAS as espécies alvo:")
for geotiff in geotiffs_with_all_target_species.index:
    print(geotiff)

mosaics = geotiffs_with_all_target_species.index

# Quantidade de amostras em cada mosaico específico
counts = [gdf[gdf["geotiff"].str.contains(mosaic)].shape[0] for mosaic in mosaics]
# Quantidade total de amostras
total_samples = sum(counts)

for i, mosaic in enumerate(mosaics):
    print(f"Quantidade de amostras em {mosaic}: {counts[i]}")
    
print(f"Quantidade total de amostras: {total_samples}")

# Quantidade de amostras por espécie
species_counts = gdf[gdf["species"].isin(target_species) & gdf["geotiff"].isin(mosaics)]["species"].value_counts()
species_counts.head(10)

In [ ]:
gdf_selected = gdf[gdf["species"].isin(target_species) & gdf["geotiff"].isin(mosaics)].copy()
gdf_selected = gdf_selected.to_crs(gdf_selected.estimate_utm_crs())


gdf_selected["area_m2"] = gdf_selected.geometry.area

gdf_selected["area_m2"].describe()

In [ ]:
# Calcula a área mínima e alguns percentis de cada espécie nas amostras selecionadas
min_areas_por_especie = gdf_selected.groupby("species")["area_m2"].min()
percentis_por_especie = gdf_selected.groupby("species")["area_m2"].quantile([0.05, 0.10, 0.25, 0.5, 0.75, 0.90, 0.95]).unstack()

print("Área mínima por espécie:")
print(min_areas_por_especie)
print("\nAlguns percentis de área por espécie:")
print(percentis_por_especie)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.boxplot(x=gdf_selected["area_m2"])
plt.xlabel("Área (m²)")
plt.title("Boxplot da área das amostras selecionadas")
plt.show()

In [ ]:
# Pegando o centroid do primeiro shapefile em gdf (assumindo que 'geometry' é a coluna de interesse)
from shapely.geometry import Point
import pyproj

primeira_geom = gdf.geometry.iloc[0]
centroide = primeira_geom.centroid

# Converter para latitude/longitude (EPSG:4326), compatível com Google Maps
# A suposição aqui é que o CRS do gdf é correto; adapte 'gdf.crs' se necessário!
project_to_wgs84 = pyproj.Transformer.from_crs(gdf.crs, "EPSG:4326", always_xy=True)
lon, lat = project_to_wgs84.transform(centroide.x, centroide.y)

print("Coordenadas do centróide (Google Maps):", (lat, lon))
